In [2]:
import pandas as pd
from google.cloud import bigquery

billing_id = "river-pollution-499210"

query = """
SELECT
    dados.ano AS ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.sexo AS sexo,
    dados.grupo_idade AS grupo_idade,
    dados.populacao AS populacao
FROM `basedosdados.br_ms_populacao.municipio` AS dados
LEFT JOIN (
    SELECT DISTINCT
        id_municipio,
        nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
"""

client = bigquery.Client(project=billing_id)

job_config = bigquery.QueryJobConfig(
    use_legacy_sql=False
)

df = client.query(
    query,
    job_config=job_config,
    project=billing_id,
).to_dataframe(
    create_bqstorage_client=True
)

df.head()

,ano,id_municipio,id_municipio_nome,sexo,grupo_idade,populacao
0,2000,1703701,Brejinho de Nazaré,feminino,0-4 anos,286
1,2000,3119609,Coronel Pacheco,feminino,0-4 anos,120
2,2000,4200606,Águas Mornas,feminino,0-4 anos,246
3,2000,4211751,Otacílio Costa,feminino,0-4 anos,760
4,2000,4214151,Princesa,feminino,0-4 anos,120


In [7]:
import re
import unicodedata

def normalize_text(s: str) -> str:
    if pd.isna(s):
        return s

    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


pop = (
    df
    .rename(columns={
        "ano": "year",
        "id_municipio": "mun_id",
        "id_municipio_nome": "mun_name",
        "sexo": "sex",
        "grupo_idade": "age_group",
        "populacao": "population",
    })
    .assign(
        mun_id=lambda d: d["mun_id"].astype(str).str[:6],
        sex=lambda d: (
            d["sex"]
            .map(normalize_text)
            .replace({
                "feminino": "female",
                "masculino": "male",
            })
        ),
        age_group=lambda d: (
            d["age_group"]
            .astype(str)
            .str.replace(" anos", "", regex=False)
            .str.replace("80-mais", "80_plus", regex=False)
            .map(normalize_text)
        ),
        population=lambda d: pd.to_numeric(d["population"], errors="coerce"),
        year=lambda d: pd.to_numeric(d["year"], errors="coerce").astype("Int64"),
    )
    .drop(columns=["mun_name"])
    .loc[:, ["mun_id", "year", "sex", "age_group", "population"]]
)

pop.head()

,mun_id,year,sex,age_group,population
0,170370,2000,female,0_4,286
1,311960,2000,female,0_4,120
2,420060,2000,female,0_4,246
3,421175,2000,female,0_4,760
4,421415,2000,female,0_4,120
